In [1]:
print("Hello, world!")

Hello, world!


In [1]:
import os
user_id = os.getenv("DANELFIN_USER_ID")

from ramlytics_database import DynamoDBClient
db = DynamoDBClient()

instruments = db.list_instruments()

accounts = db.list_accounts(user_id)


In [6]:
import pandas as pd

instruments_df = pd.DataFrame(instruments)
instruments_df
symbol = "GLD"
current_price = instruments_df[instruments_df["symbol"] == symbol]["current_price"].iloc[0]

In [7]:
current_price

Decimal('399.72')

In [12]:
import pandas as pd

def get_account_positions(user_id):
    
    accounts = db.list_accounts(user_id)
    if not accounts:
        raise ValueError("User has no accounts.")
    for account in accounts:
        account_id = account["account_id"]

        # loop through positions for the current account and collect them in a list
        positions_list = []
        for position in db.list_positions(account_id):
            symbol = position["symbol"]
            current_price = instruments_df[instruments_df["symbol"] == symbol]["current_price"].iloc[0]
            rec = {
                "account_id": account_id,
                "symbol": symbol,
                "quantity": position["quantity"],
                "avg_price_paid": position["holding"]["Price_Paid_Amt"],
                "current_price": current_price, 
            }
            positions_list.append(rec)

        # convert positions_list to DataFrame and merge with positions_df
        if positions_list:
            positions_df = pd.merge(positions_df, pd.DataFrame(positions_list), how="outer") if 'positions_df' in locals() else pd.DataFrame(positions_list)    


    return positions_df

positions_df = get_account_positions(user_id)
positions_df

# get unique symbols
unique_symbols = set(positions_df["symbol"])
unique_symbols

{'FSTA',
 'GLD',
 'HODL',
 'IDHQ',
 'IVW',
 'JNJ',
 'LVHI',
 'QQQ',
 'RYCEY',
 'SLV',
 'SPY',
 'VYM',
 'WMT',
 'XLK'}

In [ ]:
positions_df


,account_id,symbol,quantity,avg_price_paid,current_price
0,RAM BRKG -3543,GLD,5,389.1,399.72
1,RAM BRKG -3543,XLK,270,57.6495,187.87
2,RAM ROLL -1969,FSTA,30,52.8983,52.83
3,RAM ROLL -1969,GLD,5,345.476,399.72
4,RAM ROTH -2359,FSTA,10,55.8,52.83
5,RAM ROTH -2359,GLD,50,260.2822,399.72
6,RAM ROTH -2359,HODL,10,22.185,22.18
7,RAM ROTH -2359,IDHQ,200,45.2287,44.78
8,RAM ROTH -2359,IVW,80,37.1,139.82
9,RAM ROTH -2359,LVHI,75,42.77,42.45


In [27]:
# summarize accounts by symbol
def summarize_accounts_by_symbol(positions_df):
    symbol_summary_df = positions_df.groupby("symbol").agg(
        total_quantity=pd.NamedAgg(column="quantity", aggfunc="sum"),
        avg_price_paid=pd.NamedAgg(column="avg_price_paid", aggfunc="mean"),
        current_price=pd.NamedAgg(column="current_price", aggfunc="mean"),
        # list of accounts
        accounts=pd.NamedAgg(column="account_id", aggfunc=lambda x: list(set(x)))
    ).reset_index()
    symbol_summary_df['total_value'] = symbol_summary_df['total_quantity'].astype(float) * symbol_summary_df['current_price'].astype(float)
    return symbol_summary_df

summarize_accounts_by_symbol(positions_df)

,symbol,total_quantity,avg_price_paid,current_price,accounts,total_value
0,FSTA,369,53.2459,52.83,"[RAM ROTH -2359, SFM ROLL -8322, SFM ROTH -223...",19494.27
1,GLD,105,358.62064,399.72,"[RAM ROTH -2359, SFM ROTH -2236, SFM ROLL -832...",41970.60
2,HODL,10,22.185,22.18,[RAM ROTH -2359],221.80
3,IDHQ,700,45.25185,44.78,"[RAM ROTH -2359, SFM ROLL -8322]",31346.00
4,IVW,250,83.56545,139.82,"[RAM ROTH -2359, SFM ROLL -8322]",34955.00
5,JNJ,50,162.0,275.23,[SFM ROLL -8322],13761.50
6,LVHI,1150,42.798633,42.45,"[RAM ROTH -2359, SFM ROLL -8322, SFM ROTH -2236]",48817.50
7,QQQ,85,341.28835,718.36,"[RAM ROTH -2359, SFM ROLL -8322]",61060.60
8,RYCEY,144,11.68845,19.97,"[RAM ROTH -2359, SFM ROLL -8322]",2875.68
9,SLV,85,53.1956,59.82,"[SFM ROLL -8322, SFM ROTH -2236]",5084.70


In [ ]:
from ramlytics_yfinance import yf_search
yf_search("AAPL")

In [ ]:
# local test for dyndb_load_json module

import backend.dyndb_load_json.dyndb_load_json as dyndb_load_json

import os
outpath = os.getenv("OUTPATH", "/tmp/accounts") 

import boto3
s3_bucket = os.getenv("S3_BUCKET", "fh-danelfin-289755104220")
s3_key_prefix = "ingest/market_data/"
s3_client = boto3.client("s3")

# for each .json in the s3 bucket under the key prefix
response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_key_prefix)
for obj in response.get("Contents", []):
    key = obj["Key"]

    # process only files in root folder
    if "/" in key[len(s3_key_prefix):]:
        continue

    if key.endswith(".json") :
        #print(f"Found JSON file: s3://{s3_bucket}/{key}")

        # emulate an s3 ObjectCreated event for the lambda handler
        event = {
            "Records": [
                {
                    "s3": {
                        "bucket": {"name": s3_bucket},
                        "object": {"key": key},
                    }
                }
            ]
        }
        dyndb_load_json.lambda_handler(event, None) # local test

print("Finished processing all JSON files.")



Next:  
1.  package and deploy dyndb_load_json as a lambda function
    -   why no .venv in the folder?

2.  does danelfin need repackaged?  






In [ ]:
danelfin_api_key = os.getenv("DANELFIN_API_KEY")

import backend.danelfin.danelfin as danelfin
client = danelfin.DanelfinClient(api_key=danelfin_api_key)

from datetime import datetime

# 1. Historical data for a ticker
pypl_history = client.ranking(ticker="XLK")
pypl_history
